# Setup

Transformers not needed

Focus is on running vLLM service, not inspecting tokenizer


In [1]:
import time
from pprint import pprint

import httpx


VLLM_BASE_URL = "http://127.0.0.1:8000/v1"


# Discover whatever model your local server exposes.
response = httpx.get(
    f"{VLLM_BASE_URL}/models",
    timeout=10.0,
)

response.raise_for_status()

models = response.json()["data"]

for model in models:
    print(model["id"])

SERVED_MODEL = models[0]["id"]

print("\nUsing model:")
print(SERVED_MODEL)

Qwen/Qwen3.5-0.8B

Using model:
Qwen/Qwen3.5-0.8B


### Setup Inference Helper

Use 1 function across experiment so measurement method remains consistent


In [2]:
def run_chat(
    prompt: str,
    *,
    temperature: float,
    top_p: float,
    top_k: int,
    max_tokens: int = 128,
    presence_penalty: float = 0.0,
    seed: int | None = None,
    enable_thinking: bool = False
):
    # payload conforming to OpenAI API Format
    # JSON Structure & URL endpoints OpenAI created for ChatGPT
    payload = {
        "model": SERVED_MODEL,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
        "presence_penalty": presence_penalty, # penalize repeating same words / topics etc.
        "max_tokens": max_tokens,
        "chat_template_kwargs": {
            "enable_thinking": enable_thinking
        },
    }
    
    if seed is not None:
        payload["seed"] = seed
    
    start = time.perf_counter()
    
    res = httpx.post(
        f"{VLLM_BASE_URL}/chat/completions",
        json=payload,
        timeout=60.
    )
    
    latency_ms = (time.perf_counter() - start) * 1000
    res.raise_for_status()
    
    body = res.json()
    
    choice = body["choices"][0] # select top choice
    message = choice["message"]
    
    return {
        "content": message.get("content"),
        "reasoning": message.get("reasoning"),
        "finish_reason": choice.get("finish_reason"),
        "prompt_tokens": body["usage"].get("prompt_tokens"),
        "completion_tokens": body["usage"].get("completion_tokens"),
        "total_tokens": body["usage"].get("total_tokens"),
        "latency_ms": round(latency_ms, 2),
    }

## Experiment A - Determinism & Seed

Qns -> If we send the exact same prompt repeatedly, do we always get the same answer?


In [3]:
PROMPT_A = """
Create one short codename for an AI incident investigation platform.
Return only the codename.
""".strip()

In [4]:
# A1 - Temp 0 - Greedy - always select best
greedy_results = []

for i in range(5):
    result = run_chat(
        PROMPT_A,
        temperature=0.0,
        top_p=1.0,
        top_k=20,
        enable_thinking=False,
    )

    greedy_results.append(result)

    print(
        f"Run {i + 1}: "
        f"{result['content']!r}"
    )

Run 1: 'A.I. Incident'
Run 2: 'A.I. Incident'
Run 3: 'A.I. Incident'
Run 4: 'A.I. Incident'
Run 5: 'A.I. Incident'


In [6]:
# A2 - Stochastic Sampling (random) temp = 1
sampled_results = []

for i in range(5):
    result = run_chat(
        PROMPT_A,
        temperature=1.0,
        top_p=1.0,
        top_k=20,
        enable_thinking=False,
    )

    sampled_results.append(result)

    print(
        f"Run {i + 1}: "
        f"{result['content']!r}"
    )

Run 1: 'NEO-VOID'
Run 2: 'Nexus'
Run 3: 'IncidentScan'
Run 4: 'A.I.S.D.A.L.E.'
Run 5: 'Aegis'


In [11]:
# A3 - Same Seed

seeded_results = []

for i in range(5):
    result = run_chat(
        PROMPT_A,
        temperature=1,
        top_p=1.0,
        top_k=20,
        seed=42,
        enable_thinking=False,
    )

    seeded_results.append(result)

    print(
        f"Run {i + 1}: "
        f"{result['content']!r}"
    )

Run 1: 'A-07'
Run 2: 'A-07'
Run 3: 'A-07'
Run 4: 'A-07'
Run 5: 'A-07'
